# 3일차 파트 3 실습: 첫 RAG 완성

실습은 하나입니다(실습 3). 검색 결과로 프롬프트를 만드는 함수와 구판 청크를 제외하는 함수를 직접 작성하고, 새 문서 묶음으로 RAG를 돌려 봅니다.
`NotImplementedError`가 있는 곳을 직접 채웁니다. 막히면 힌트를 펼쳐 보고, 다 한 뒤에 `3일차_03_실습_풀이.ipynb`와 비교합니다.
테스트 셀은 임베딩 모델도 API도 부르지 않고 작은 입력으로 함수를 검사합니다. 준비 셀의 연결 확인을 빼면, 마지막 세 셀에서만 모델을 호출합니다(모두 여덟 번).

## 준비

이 노트북은 `노트북` 폴더에서 실행합니다. 같은 폴더의 `llm.py`가 API 호출을 맡습니다. Python 3.10 이상이 필요하고, 패키지는 다음 명령으로 설치합니다.

```bash
python -m pip install anthropic openai pandas pydantic openpyxl jupyter
```

일자마다 더 필요한 패키지가 있으면 아래 "준비 (2)" 셀 안의 `# 설치:` 주석에 명령이 있습니다. 그 명령도 실행합니다.

### API 키는 코드에 적지 않습니다

API 키는 API를 호출한 사람을 식별하는 문자열입니다. 사용 기록과 요금은 키별로 집계됩니다. 키를 가진 사람은 누구나 그 키로 호출할 수 있고, 요금은 키의 주인에게 청구됩니다.

코드에 적은 키는 파일을 공유하거나 저장소에 올리는 순간 함께 퍼집니다. 그래서 키는 환경 변수에 두고 코드에서는 환경 변수를 읽습니다.

> **주의**
>
> 키를 메신저, 화면 공유, 캡처 이미지에 노출하지 않습니다. 노출됐다면 바로 폐기하고 새로 발급받습니다.

### 환경 변수 설정

`llm.py`는 아래 환경 변수를 읽어 어느 API로 보낼지 고릅니다.

**Anthropic**

| 환경 변수 | 값 |
| --- | --- |
| `ANTHROPIC_API_KEY` | 발급받은 키 |

모델 이름은 설정하지 않아도 됩니다. 소형, 기본, 대형 모델 등급에 현행 Claude 모델이 기본값으로 들어 있습니다.

**사내 API(OpenAI 호환)**

| 환경 변수 | 값 |
| --- | --- |
| `LLM_PROVIDER` | `openai` |
| `OPENAI_BASE_URL` | 사내 API 주소. 보통 `/v1`로 끝납니다 |
| `OPENAI_API_KEY` | 사내 API 키 |
| `LLM_MODEL_SMALL` | 소형 모델 이름 |
| `LLM_MODEL_DEFAULT` | 기본 모델 이름 |
| `LLM_MODEL_LARGE` | 대형 모델 이름 |
| `LLM_STRUCTURED` | 구조화 출력을 요청하는 방식. 설정하지 않으면 `json_schema`입니다. 서버가 이 방식을 지원하지 않을 때만 바꿉니다(2일차 파트 2의 "구조화 출력" 절) |

사내에서 쓸 수 있는 모델이 하나뿐이면 세 변수에 같은 이름을 넣습니다. 그러면 모델 등급을 비교하는 예제와 실습에서는 차이가 나지 않습니다.

환경 변수는 Jupyter를 시작하기 전에 설정합니다. `setx`로 설정한 값은 그 뒤에 새로 연 창에만 적용됩니다. Windows PowerShell에서 다음과 같이 설정한 뒤 터미널을 새로 열고, 그 터미널에서 Jupyter를 시작합니다. VS Code로 노트북을 연다면 VS Code 창을 모두 닫았다가 다시 엽니다.

Anthropic API를 쓸 때:

```powershell
setx ANTHROPIC_API_KEY "발급받은_키"
```

사내 API를 쓸 때(위 표의 나머지 변수도 같은 방법으로 설정합니다):

```powershell
setx LLM_PROVIDER openai
setx OPENAI_BASE_URL "사내_API_주소"
setx OPENAI_API_KEY "사내_API_키"
```

`setx`로 설정해 두면 모든 일자의 노트북에서 그대로 씁니다.

환경 변수를 미리 설정하지 못했다면 아래 "준비 (1)" 셀이 키, 주소, 모델 이름을 물어봅니다. 입력한 값은 화면에 보이지 않고 노트북 파일에도 저장되지 않습니다. 이 값은 커널이 살아 있는 동안만 쓰이므로, 노트북을 새로 열거나 커널을 다시 시작하면 다시 입력합니다. 단, 사내 API를 쓰려면 `LLM_PROVIDER=openai`만은 Jupyter를 시작하기 전에 설정해야 합니다. "준비 (1)" 셀이 이 값을 보고 어떤 값을 물을지 정하기 때문입니다. 이 값이 없으면 `llm.py`는 Anthropic API로 요청을 보냅니다.

### 연결 확인

아래 두 셀을 차례로 실행합니다. 두 셀이 오류 없이 끝나고 마지막 줄에 모델의 응답(보통 `준비 완료`)이 나오면 연결된 것입니다. 응답 문구는 모델에 따라 조금 다를 수 있습니다.

In [ ]:
# 준비 (1) 환경 변수가 비어 있으면 여기서 입력합니다
import os
from getpass import getpass

import llm

needed = (['ANTHROPIC_API_KEY'] if llm.provider() == 'anthropic'
          else ['OPENAI_BASE_URL', 'OPENAI_API_KEY', 'LLM_MODEL_SMALL', 'LLM_MODEL_DEFAULT',
                'LLM_MODEL_LARGE'])
for name in needed:
    if not os.environ.get(name):
        os.environ[name] = getpass(f'{name}: ').strip()
print('사용하는 API:', llm.provider())

In [ ]:
# 준비 (2) 공통으로 쓰는 것을 불러오고 연결을 확인합니다
import json
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

# 설치: python -m pip install sentence-transformers scikit-learn pypdf python-docx openpyxl pandas
import numpy as np

import embed

DATA = 'data/3일차'
CORPUS = 'data/2일차/corpus'      # 2일차와 같은 문서 253건
OUT = 'out'                        # 예제가 만드는 파일을 두는 폴더
os.makedirs(OUT, exist_ok=True)


for alias in ['소형', '기본', '대형']:
    try:
        print(f'{alias} 모델:', llm.model_name(alias))
    except RuntimeError:
        print(f'{alias} 모델: 정하지 않음')

reply = llm.send([{'role': 'user', 'content': '"준비 완료"라고만 답해 주세요.'}], model='소형', max_tokens=200)
print(reply.text)

오류가 나면 아래 표에서 원인을 찾습니다.

| 나온 문구 | 원인과 조치 |
| --- | --- |
| `환경 변수 ...의 값이 비어 있습니다` | 환경 변수를 설정하지 않았습니다. 변수 이름의 철자를 확인하고 "준비 (1)" 셀을 다시 실행합니다 |
| `환경 변수 LLM_MODEL_...에 '소형' 모델 이름을 넣어 주세요` | 사내 API를 쓰는데 모델 이름을 비워 두었습니다. 커널을 다시 시작하고 "준비 (1)" 셀에서 모델 이름을 입력합니다 |
| `401 auth` | 키가 틀렸습니다. "준비 (1)" 셀은 값이 비어 있을 때만 묻습니다. 셀에서 입력했다면 커널을 다시 시작한 뒤 "준비 (1)" 셀에서 앞뒤 공백이나 따옴표 없이 다시 입력합니다. `setx`로 설정했다면 값을 다시 설정하고 Jupyter를 다시 시작합니다 |
| `403 permission` | 키에 그 모델이나 기능을 쓸 권한이 없습니다. 키를 발급한 곳에 권한을 확인합니다 |
| `404 not_found` | 모델 이름이나 API 주소가 틀렸습니다. 고치는 방법은 `401 auth`와 같습니다 |
| `connection` | 서버에 연결하지 못했습니다. 주소와 사내망 연결을 확인합니다 |
| `timeout` | 정해진 시간 안에 응답이 오지 않았습니다. 잠시 뒤 다시 실행합니다 |
| `NameError` | 앞 셀을 건너뛰었거나 커널을 다시 시작했습니다. 위에서부터 차례로 다시 실행합니다 |

## 실습 준비

아래 셀은 실습이 쓰는 재료와 함수를 만듭니다. 그대로 실행합니다. 교재 예제 3-1의 함수(`read_manifest`, `build_index`, `search`, `show`)와 새 문서 묶음의 인덱스를 만듭니다.

| 이름 | 내용 |
| --- | --- |
| `Chunk`, `Hit`, `ChunkIndex` | 청크, 검색 결과 한 건, 청크 목록과 벡터를 묶은 인덱스입니다 |
| `search(index, question, k)` | 질문과 가까운 청크 k건을 `Hit` 목록으로 돌려줍니다 |
| `with_titles(index)` | 청크 앞에 문서 제목을 붙여 다시 임베딩한 인덱스를 돌려줍니다(교재 예제 3-7) |
| `RULE` | 근거 없음 규칙의 문장입니다 |
| `base` | 새 문서 12건의 청크 인덱스(39청크)입니다. 문서 제목을 붙여 임베딩했고, 구판은 들어 있지 않습니다 |

In [ ]:
# 실습 준비
import shutil
import tempfile

import checks_d3_3

import csv
import re
import unicodedata
from collections import namedtuple

Chunk = namedtuple('Chunk', 'doc_id version type date section text')
Hit = namedtuple('Hit', 'score doc_id version type date section text')
ChunkIndex = namedtuple('ChunkIndex', 'chunks vectors')


def read_manifest(path):
    """문서 목록 파일을 {파일 이름: 메타데이터}로 읽습니다."""
    with open(path, encoding='utf-8-sig') as file:
        return {unicodedata.normalize('NFC', row['file']): row for row in csv.DictReader(file)}


def build_chunks(folder, manifest):
    """문서를 '## ' 절 단위로 잘라 메타데이터를 붙인 Chunk 목록을 돌려줍니다."""
    chunks = []
    for raw in sorted(os.listdir(folder)):
        file = unicodedata.normalize('NFC', raw)         # 한글 파일 이름을 목록 파일의 이름과 같은 형태로 맞춥니다
        if not file.endswith('.md'):
            continue
        meta = manifest[file]
        head, *sections = re.split(r'\n(?=## )', open(f'{folder}/{raw}', encoding='utf-8').read())
        for part in [head] + sections:
            part = part.strip()
            if not part:
                continue
            if sections and part == head.strip():
                part = '\n'.join(part.splitlines()[:8])      # 첫 조각은 제목과 문서 정보 줄까지만 둡니다
            title = part.splitlines()[0].lstrip('# ').strip()
            chunks.append(Chunk(meta['doc_id'], meta['version'], meta['type'], meta['date'], title, part[:1200]))
    return chunks


def build_index(folder, manifest):
    chunks = build_chunks(folder, manifest)
    return ChunkIndex(chunks, embed.passages([chunk.text for chunk in chunks]))


def search(index, question, k=4, exclude_types=()):
    """질문과 가까운 청크를 점수가 높은 순으로 k건 찾습니다. exclude_types에 든 유형의 문서는 뺍니다."""
    allowed = [row for row, chunk in enumerate(index.chunks) if chunk.type not in exclude_types]
    scores = index.vectors[allowed] @ embed.query(question)
    return [Hit(round(float(scores[row]), 3), *index.chunks[allowed[row]])
            for row in np.argsort(-scores, kind='stable')[:k]]


def show(hits):
    for hit in hits:
        print(f'  {hit.score:.3f} {hit.doc_id}_{hit.version} ({hit.section[:22]})')



def with_titles(index):
    """청크 앞에 문서 제목을 붙여 다시 임베딩한 인덱스를 돌려줍니다. 청크의 본문은 그대로입니다."""
    title_of = {}
    for chunk in index.chunks:
        title_of.setdefault((chunk.doc_id, chunk.version), chunk.section)      # 문서의 첫 청크의 제목이 문서 제목입니다
    return ChunkIndex(index.chunks, embed.passages(
        [f'{title_of[chunk.doc_id, chunk.version]} · {chunk.section}\n{chunk.text}' for chunk in index.chunks]))


RULE = "검색 결과에 없으면 '사내 문서에서 근거를 찾지 못했습니다' 라고만 답한다."
manifest_m3 = read_manifest(f'{DATA}/corpus_m3_manifest.csv')
base = with_titles(build_index(f'{DATA}/corpus_m3', manifest_m3))
print(f"새 문서 {len(os.listdir(f'{DATA}/corpus_m3'))}건, 청크 {len(base.chunks)}, 인덱스 {base.vectors.shape}")

## 실습 3 새 문서 묶음으로 RAG 구성하고 구판 제외하기

**목적** 검색 결과를 프롬프트로 만드는 일과, 구판 청크를 제외하는 일을 직접 작성합니다. 근거 없음 규칙을 뺐을 때와 구판을 넣었을 때 응답의 내용과 값이 어떻게 바뀌는지 새 문서에서 확인합니다.

**재료** `data/3일차/corpus_m3/`(확산, 이온주입 라인의 새 문서 12건), `data/3일차/corpus_m3_구판/SOP-DIFF-002_v1.0.md`(구판 1건), `data/3일차/실습_질문_5.csv`(질문 5개. 5번은 일부러 답이 없는 질문입니다).

**할 일** 함수 두 개를 작성합니다.

1. `build_prompt(hits, question, use_rule=True)`: 검색 결과 `hits`(`Hit` 목록)와 질문으로 프롬프트 글을 만듭니다.
   - 교재 예제 3-3의 `TEMPLATE`과 같은 네 요소(역할 지정, 검색 결과, 출력 규칙 "문장마다 근거 [문서번호]를 붙인다", 근거 없음 규칙)로 프롬프트를 구성합니다. 이후 셀이 이 함수로 만든 프롬프트로 LLM API를 호출하므로 출력 규칙도 넣습니다.
   - 청크마다 `[문서번호_버전] (절)` 꼴의 머리글을 앞에 붙여 순서대로 넣습니다. 예: `[SOP-DIFF-001_v1.0] (절차)`
   - `use_rule`이 참이면 근거 없음 규칙 `RULE`을 검색 결과 뒤, 질문 앞에 넣습니다. 거짓이면 넣지 않습니다.
   - 질문이 프롬프트의 맨 끝에 옵니다.
2. `latest_version_rows(chunks)`: `Chunk` 목록에서 같은 문서 번호(`doc_id`)의 청크 가운데 버전이 가장 높은 청크의 행 번호를 돌려줍니다. 버전은 `v1.10`이 `v1.9`보다 높게 숫자로 비교합니다. 판이 하나뿐인 문서는 그대로 남깁니다.

**완료 기준** 테스트 셀이 "실습 3 통과"를 출력합니다. 그다음 세 셀을 실행해 아래 "적어 보기"를 채웁니다.

<details><summary>힌트 1</summary>

`build_prompt`의 머리글은 `f'[{hit.doc_id}_{hit.version}] ({hit.section})\n{hit.text}'`입니다. 청크 사이는 빈 줄로 구분합니다.
</details>

<details><summary>힌트 2</summary>

버전 `'v1.10'`은 `tuple(int(number) for number in version.lstrip('v').split('.'))`로 `(1, 10)`이 됩니다. 튜플끼리는 `(1, 10) > (1, 9)`처럼 비교됩니다. 문자열로 비교하면 `'v1.9' > 'v1.10'`이 됩니다.
</details>

<details><summary>힌트 3</summary>

문서 번호별로 가장 높은 버전을 먼저 구해 두고, 청크를 한 번 더 훑으며 그 버전인 청크의 행 번호를 모읍니다.
</details>

In [ ]:
# 실습 3
def build_prompt(hits, question, use_rule=True):
    """검색 결과와 질문으로 프롬프트 글을 만듭니다."""
    raise NotImplementedError('이 부분을 채웁니다')


def latest_version_rows(chunks):
    """같은 문서 번호의 청크 가운데 버전이 가장 높은 청크의 행 번호 목록을 돌려줍니다."""
    raise NotImplementedError('이 부분을 채웁니다')

In [ ]:
# 실습 3 확인 (임베딩 모델도 API도 부르지 않습니다)
checks_d3_3.check_practice_3(build_prompt, latest_version_rows, RULE)

In [ ]:
# 실습 3 (1) 새 문서의 질문 다섯 개를 RAG로 처리합니다
practice_questions = list(pd.read_csv(f'{DATA}/실습_질문_5.csv', encoding='utf-8-sig')['질문'])


def answer(index, question, use_rule=True, k=4):
    """검색하고 build_prompt()로 프롬프트를 만들어 모델에 보냅니다. (응답 글, 검색 결과)를 돌려줍니다."""
    hits = search(index, question, k)
    return llm.ask(build_prompt(hits, question, use_rule), model='기본', max_tokens=600).text, hits


for number, question in enumerate(practice_questions, 1):
    text, hits = answer(base, question)
    print(f'{number}. {question}')
    show(hits)
    print('   응답:', text[:130].replace('\n', ' '))

In [ ]:
# 실습 3 (2) 5번 질문(답이 없음)을 근거 없음 규칙 없이 처리합니다
text, hits = answer(base, practice_questions[4], use_rule=False)
print('규칙 없음, 검색 결과 최고 점수', hits[0].score)
print('응답:', text[:250].replace('\n', ' '))

In [ ]:
# 실습 3 (3) 구판을 인덱스에 넣었을 때와 뺐을 때
QUESTION_VERSION = '확산로 온도 프로파일 허용 편차는?'
with tempfile.TemporaryDirectory() as folder:                 # 원본 폴더는 건드리지 않고 임시 폴더에 구판을 함께 둡니다
    for source in [f'{DATA}/corpus_m3', f'{DATA}/corpus_m3_구판']:
        for file in os.listdir(source):
            shutil.copy(f'{source}/{file}', folder)
    with_old = with_titles(build_index(folder, manifest_m3))
print(f'청크 {len(base.chunks)}개 → 구판을 넣으면 {len(with_old.chunks)}개')

rows = latest_version_rows(with_old.chunks)
without_old = ChunkIndex([with_old.chunks[row] for row in rows], with_old.vectors[rows])
for label, index in [('구판 포함', with_old), ('구판 제외', without_old)]:
    text, hits = answer(index, QUESTION_VERSION)
    print(f'{label}, 검색 결과의 판 {[f"{hit.doc_id}_{hit.version}" for hit in hits]}')
    print('   응답:', text[:160].replace('\n', ' '))

**적어 보기**

- 질문 5개의 검색 결과 1위 문서와 점수. 답이 없는 5번 질문의 응답은 무엇이었는지:
- 5번 질문을 근거 없음 규칙 없이 처리했을 때 응답은 어떻게 바뀌었는지. 운영에서 무엇이 곤란해지는지:
- 구판을 넣었을 때 검색 결과 네 건에 v1.0과 v2.0이 각각 몇 건 들어왔는지, 응답의 허용 편차 값은 무엇이었는지:
- 응답이 맞게 나왔더라도 구판을 인덱스에 남겨 두면 안 되는 이유(검색 결과의 판 구성을 근거로):

## 더 해 보기

- `build_prompt()`의 규칙에 "같은 문서의 여러 판이 있으면 버전이 높은 것을 따른다"를 더해, 구판을 인덱스에 넣은 상태에서 응답이 어떻게 바뀌는지 세 번씩 비교합니다.
- 교재의 `diagnose()`를 새 문서 묶음에 적용해 질문 다섯 개를 한 표로 봅니다.
- 이 실습의 '할 일'을 AI 코딩 도구에 보여 주고 `build_prompt()`와 `latest_version_rows()`를 만들게 한 뒤 테스트 셀로 검사합니다. 어느 항목이 실패하는지 확인합니다.